# 00 · Configuración del entorno

Fase 0 del proyecto. Prepara el entorno en Google Colab, fija la semilla, descarga el dataset y verifica que coincida con la documentación de Realinho et al. (2022).

In [1]:
# Clonar el repositorio (solo en Colab). Cambiar REPO_URL por la URL real.
import os, sys
REPO_URL = "https://github.com/andresjaimes2004/tesis-desercion.git"
REPO_DIR = "/content/tesis-desercion"

if "google.colab" in sys.modules and not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Directorio de trabajo:", os.getcwd())

Cloning into '/content/tesis-desercion'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 17 (delta 2), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 7.88 KiB | 1.58 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Directorio de trabajo: /content/tesis-desercion


In [2]:
# Instalar solo lo que falte (Colab ya trae la mayoría de librerías)
%pip install -q -r requirements.txt

# Registrar las versiones exactas usadas, para reproducibilidad
!pip freeze > requirements-lock.txt
!python --version

Python 3.13.18


Si Colab pide reiniciar la sesión después de instalar, reiniciar y continuar desde la siguiente celda (ejecutando antes la primera).

In [3]:
from src import config
from src.utils import set_seed, environment_report

set_seed()
env = environment_report()
env

{'python': '3.13.18',
 'numpy': '2.4.6',
 'pandas': '2.2.3',
 'scipy': '1.16.3',
 'sklearn': '1.6.1',
 'imblearn': '0.14.2',
 'xgboost': '3.4.1',
 'shap': '0.54.1',
 'matplotlib': '3.10.0',
 'seaborn': '0.13.2',
 'joblib': '1.6.0'}

In [4]:
# Guardar el reporte del entorno para el anexo de reproducibilidad
import json
config.FIGURES.parent.mkdir(parents=True, exist_ok=True)
with open(config.FIGURES.parent / "entorno.json", "w", encoding="utf-8") as f:
    json.dump(env, f, indent=2, ensure_ascii=False)

## Descarga y verificación del dataset

In [5]:
from src.data import load_raw, binarize_target

df = load_raw()
print("Registros:", df.shape[0], "| Columnas:", df.shape[1])

Registros: 4424 | Columnas: 37


In [6]:
# Verificaciones contra la documentación publicada
print("Valores faltantes:", int(df.isna().sum().sum()))
print("Duplicados:", int(df.duplicated().sum()))
print("Predictoras:", df.shape[1] - 1)
print()
print(df[config.TARGET_COL].value_counts())
print()
print((df[config.TARGET_COL].value_counts(normalize=True) * 100).round(1))

Valores faltantes: 0
Duplicados: 0
Predictoras: 36

Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Target
Graduate    49.9
Dropout     32.1
Enrolled    17.9
Name: proportion, dtype: float64


Valores esperados según Realinho et al. (2022): 4.424 registros, sin valores faltantes, Graduate 2.209, Dropout 1.421, Enrolled 794. El número de predictoras resuelve la diferencia 35 frente a 36 señalada en la revisión.

In [7]:
df = binarize_target(df)
print(df[config.BINARY_TARGET].value_counts())
print("Proporción de deserción:", round(df[config.BINARY_TARGET].mean() * 100, 1), "%")

deserta
0    3003
1    1421
Name: count, dtype: int64
Proporción de deserción: 32.1 %


In [8]:
from src.data import features_for_moment
for m in config.MOMENTS:
    print(f"{m}: {len(features_for_moment(df, m))} variables")

matricula: 24 variables
semestre_1: 30 variables
semestre_2: 36 variables
